In [1]:
import json
import logging
from pathlib import Path

from transformers import BitsAndBytesConfig

from llm_bayesian_reasoning.estimators.true_false_lm_estimator import (
    TrueFalseLLMEstimator,
)
from llm_bayesian_reasoning.pipeline import (
    PipelineConfig,
    build_or_load_index,
    run_pipeline,
)
from llm_bayesian_reasoning.problog_models.problog_models import (
    ProblogAtom,
    ProblogFormula,
)

logging.basicConfig(level=logging.INFO)

/home/mtsatsev/University/MasterThesisDataScienceOnly/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PREPROCESSED_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/preprocessed_data/parsed_test.jsonl"
)

# Load ground truth from QUEST test.jsonl
QUEST_TEST_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/quest_data/test.jsonl"
)
DOCUMENTS_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/index_data/documents.jsonl"
)
INDEX_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/index_data/bm25_index"
)
OUTPUT_PATH = Path(
    "/home/mtsatsev/University/MasterThesisDataScienceOnly"
    "/llm_bayesian_reasoning/data/results/pipeline_results.jsonl"
)

In [3]:
with open(PREPROCESSED_PATH, "r") as f:
    knowledge_base = [json.loads(line) for line in f]
knowledge_base[0]

{'id': 1,
 'query': 'German nonlinear narrative films or based on works by Bret Easton Ellis',
 'parsed': {'atoms': ['{x} is a film',
   '{x} is a German film',
   '{x} is a nonlinear narrative film',
   '{x} is based on a work by Bret Easton Ellis'],
  'logical query': '((({x} is a film) AND ({x} is a German film) AND ({x} is a nonlinear narrative film)) OR (({x} is a film) AND ({x} is based on a work by Bret Easton Ellis)))'},
 'original_query': '<mark>German nonlinear narrative films</mark> or <mark>Films based on works by Bret Easton Ellis</mark>',
 'metadata': {'template': '_ or _',
  'domain': 'films',
  'fluency': ['Mostly Fluent: It has a few errors or it does not sound natural, but I can understand it.',
   'Mostly Fluent: It has a few errors or it does not sound natural, but I can understand it.',
   'Fluent: It is clear, and grammatically correct.'],
  'meaning': ['Different Meaning: The queries ask for different sets of items. One or more of the highlighted clauses in the o

In [4]:
data = {}

for item in knowledge_base:
    parsed = item["parsed"]
    # Normalize {x} -> {X} so ProblogAtom.to_prompt substitution works correctly
    atoms = [ProblogAtom(atom=a.replace("{x}", "{X}")) for a in parsed["atoms"]]
    problog_formula = ProblogFormula(
        formula=parsed["logical query"].replace("{x}", "{X}")
    )
    data[item["id"]] = {
        "query": item["query"],
        "atoms": atoms,
        "problog_formula": problog_formula,
    }

print(f"Loaded {len(data)} records")

Loaded 1727 records


In [5]:
data = {k: v for k, v in data.items() if k in range(0, 15)}
data

{1: {'query': 'German nonlinear narrative films or based on works by Bret Easton Ellis',
  'atoms': [ProblogAtom(atom='{X} is a film', probability=0, context=None, problog_atom_format='is_a_film'),
   ProblogAtom(atom='{X} is a German film', probability=0, context=None, problog_atom_format='is_a_german_film'),
   ProblogAtom(atom='{X} is a nonlinear narrative film', probability=0, context=None, problog_atom_format='is_a_nonlinear_narrative_film'),
   ProblogAtom(atom='{X} is based on a work by Bret Easton Ellis', probability=0, context=None, problog_atom_format='is_based_on_a_work_by_bret_easton_ellis')],
  'problog_formula': ProblogFormula(formula='((({X} is a film) AND ({X} is a German film) AND ({X} is a nonlinear narrative film)) OR (({X} is a film) AND ({X} is based on a work by Bret Easton Ellis)))', head='formula', problog_formula_format='formula({X}) :-\n    (((is_a_film({X})) ,\n     (is_a_german_film({X})) ,\n     (is_a_nonlinear_narrative_film({X}))) ; ((is_a_film({X})) ,\n 

In [6]:
ground_truth = {}
with open(QUEST_TEST_PATH, "r") as f:
    for i, line in enumerate(f, start=1):
        record = json.loads(line)
        ground_truth[i] = record["docs"]

In [7]:
retriever = build_or_load_index(
    documents_path=DOCUMENTS_PATH, index_path=INDEX_PATH, batch_size=1000, limit=5000
)
print(f"Index ready — {len(retriever.entities)} documents indexed")

INFO:llm_bayesian_reasoning.pipeline.pipeline:Loading existing BM25 index from /home/mtsatsev/University/MasterThesisDataScienceOnly/llm_bayesian_reasoning/data/index_data/bm25_index


Index ready — 5000 documents indexed


In [8]:
estimator = TrueFalseLLMEstimator.from_pretrained(
    model_name="microsoft/phi-2",
    device="cuda",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)

INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████| 2/2 [00:06<00:00,  3.40s/it]


In [9]:
config = PipelineConfig(
    top_n=50,
    top_k=10,
    batch_size=1000,
    index_path=INDEX_PATH,
    output_path=OUTPUT_PATH,
    mlflow_experiment="demo experiment",
)

output = run_pipeline(
    data=data,
    retriever=retriever,
    estimator=estimator,
    config=config,
    ground_truth=ground_truth,
)

results = output["results"]
metrics = output["metrics"]

print(f"Done. Processed {len(results)} records.\n")
print("=== Metrics ===")
for name, value in metrics.items():
    if name == "num_evaluated":
        print(f"  {name:30s} {int(value)}")
    else:
        print(f"  {name:30s} {value:.4f}")

2026/03/27 17:51:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/27 17:51:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/27 17:51:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/27 17:51:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/27 17:51:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/27 17:51:50 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/27 17:51:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/27 17:51:50 INFO mlflow.store.db.utils: Updating database tables
2026/03/27 17:51:50 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 17:51:50 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/27 17:51:50 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 17:51:50 INFO alembic.runtime

Done. Processed 14 records.

=== Metrics ===
  P@1                            0.0714
  P@10                           0.0214
  R@1                            0.0060
  R@10                           0.0167
  F1@1                           0.0110
  F1@10                          0.0187
  NDCG@1                         0.0714
  NDCG@10                        0.0306
  MRR                            0.0804
  num_evaluated                  14
  avg_atoms_per_query_entity     4.7143
  total_llm_forward_passes       660.0000


In [10]:
# Launch the MLflow UI to inspect tracked runs.
# Open http://127.0.0.1:5000 in your browser after running this cell.
import subprocess
import sys

subprocess.Popen([sys.executable, "-m", "mlflow", "ui"])
print("MLflow UI started at http://127.0.0.1:5000")

MLflow UI started at http://127.0.0.1:5000


Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.


2026/03/27 17:58:54 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/27 17:58:54 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/27 17:58:54 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/27 17:58:54 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/27 17:58:54 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/27 17:58:54 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/27 17:58:54 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/27 17:58:54 INFO mlflow.store.db.utils: Updating database tables
2026/03/27 17:58:54 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 17:58:54 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/27 17:58:54 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 17:58:54 INFO alembic.runtime

INFO:     127.0.0.1:57326 - "GET / HTTP/1.1" 304 Not Modified


2026/03/27 18:01:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/27 18:01:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/27 18:01:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/27 18:01:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/27 18:01:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/27 18:01:19 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/27 18:01:19 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/27 18:01:19 INFO mlflow.store.db.utils: Updating database tables
2026/03/27 18:01:19 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 18:01:19 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/27 18:01:19 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 18:01:19 INFO alembic.runtime

...

INFO:     127.0.0.1:57334 - "GET /ajax-api/2.0/mlflow/traces?experiment_ids=2&order_by=timestamp_ms%20DESC&max_results=1&filter= HTTP/1.1" 200 OK
INFO:     127.0.0.1:57350 - "GET /ajax-api/2.0/mlflow/gateway-proxy?gateway_path=api%2F2.0%2Fendpoints%2F HTTP/1.1" 200 OK
INFO:     127.0.0.1:57334 - "POST /ajax-api/2.0/mlflow/logged-models/search HTTP/1.1" 200 OK


2026/03/27 18:01:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/03/27 18:01:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/03/27 18:01:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/03/27 18:01:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/03/27 18:01:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/03/27 18:01:23 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/03/27 18:01:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/27 18:01:23 INFO mlflow.store.db.utils: Updating database tables
2026/03/27 18:01:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 18:01:23 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/27 18:01:23 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 18:01:23 INFO alembic.runtime

INFO:     127.0.0.1:57360 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK


2026/03/27 18:01:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/27 18:01:28 INFO mlflow.store.db.utils: Updating database tables
2026/03/27 18:01:28 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 18:01:28 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/03/27 18:01:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/27 18:01:28 INFO mlflow.store.db.utils: Updating database tables
2026/03/27 18:01:28 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/03/27 18:01:28 INFO alembic.runtime.migration: Will assume non-transactional DDL.


INFO:     127.0.0.1:45982 - "GET /ajax-api/2.0/mlflow/model-versions/search?filter=run_id%3D%27210c58756f564f59be04a5c5cb1f0fb2%27 HTTP/1.1" 200 OK
INFO:     127.0.0.1:57360 - "POST /graphql HTTP/1.1" 200 OK
INFO:     127.0.0.1:45982 - "GET /ajax-api/2.0/mlflow/model-versions/search?filter=tags.%60mlflow.prompt.is_prompt%60+%3D+%27true%27+AND+tags.%60mlflow.prompt.run_ids%60+ILIKE+%22%25210c58756f564f59be04a5c5cb1f0fb2%25%22 HTTP/1.1" 200 OK
INFO:     127.0.0.1:57360 - "POST /ajax-api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:45986 - "GET /ajax-api/2.0/mlflow/metrics/get-history-bulk-interval?run_ids=210c58756f564f59be04a5c5cb1f0fb2&metric_key=avg_candidate_score&max_results=320 HTTP/1.1" 200 OK
INFO:     127.0.0.1:45986 - "GET /ajax-api/2.0/mlflow/metrics/get-history-bulk-interval?run_ids=210c58756f564f59be04a5c5cb1f0fb2&metric_key=num_candidates&max_results=320 HTTP/1.1" 200 OK
INFO:     127.0.0.1:45986 - "GET /ajax-api/2.0/mlflow/metrics/get-history-bulk-interval?r